# DataFrame 모양(shape) 변경

wide(columns)   <----> long(rows)

In [27]:
import numpy as np
import pandas as pd
import seaborn as sns

# stack vs unstack

In [2]:
df= pd.DataFrame(data=np.arange(1,7).reshape((2,3)),
                 columns=['a','b','c'],
                 index=['X','Y'])
df

,a,b,c
X,1,2,3
Y,4,5,6


In [3]:
df_stacked =df.stack() # stack: wide(columns) - long(rows)
df_stacked # 컬럼 이름들은 index로 바뀜.

X  a    1
   b    2
   c    3
Y  a    4
   b    5
   c    6
dtype: int64

In [5]:
df

,a,b,c
X,1,2,3
Y,4,5,6


In [6]:
df_unstack=df_stacked.unstack()  # unstack:long(rows) >> wide(columns)
df_unstack # 가장 마지막 계층(level)의 인덱스를 컬럼으로 변환.  level=-1(기본값)

,a,b,c
X,1,2,3
Y,4,5,6


In [7]:
df_stacked.unstack(level=0)

,X,Y
a,1,4
b,2,5
c,3,6


## 컬럼 이름이 multi-level 인덱스인 경우

In [10]:
df=pd.DataFrame(data=np.arange(1,13).reshape(2,6),
                columns=[['Fri']*2+['Sat']*2+['Sun']*2,['Lunch','Dinner']*3],
                )
df

Fri          Sat          Sun       
  Lunch Dinner Lunch Dinner Lunch Dinner
0     1      2     3      4     5      6
1     7      8     9     10    11     12

In [12]:
df.columns

MultiIndex([('Fri',  'Lunch'),
            ('Fri', 'Dinner'),
            ('Sat',  'Lunch'),
            ('Sat', 'Dinner'),
            ('Sun',  'Lunch'),
            ('Sun', 'Dinner')],
           )

In [14]:
df.stack(future_stack=True) # level=-1(기본값): 가장 마지막 레벨의 컬럼 이름들을 인덱스(row)로 변환.

Fri  Sat  Sun
0 Lunch     1    3    5
  Dinner    2    4    6
1 Lunch     7    9   11
  Dinner    8   10   12

In [15]:
df.stack(level=0,future_stack=True) # 첫번째 레벨의 컬럼 이름들을 인덱스로 변환.

Lunch  Dinner
0 Fri      1       2
  Sat      3       4
  Sun      5       6
1 Fri      7       8
  Sat      9      10
  Sun     11      12

# pivot vs melt

In [17]:
df= pd.DataFrame(data={
    'time': ['Lunch']*3+['Dinner']*3,
    'day': ['Fri','Sat','Sun']*2,
    'tip': np.arange(1,7),
    'total_bill': np.arange(10,70,10)
})
df

,time,day,tip,total_bill
0,Lunch,Fri,1,10
1,Lunch,Sat,2,20
2,Lunch,Sun,3,30
3,Dinner,Fri,4,40
4,Dinner,Sat,5,50
5,Dinner,Sun,6,60


In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   time        6 non-null      object
 1   day         6 non-null      object
 2   tip         6 non-null      int64 
 3   total_bill  6 non-null      int64 
dtypes: int64(2), object(2)
memory usage: 324.0+ bytes


## pivot

**카테고리 타입** 컬럼의 값들이 컬럼 이름 또는 (row) 인덱스로 변환.

`pd.DataFrame.pivot()` 메서드 파라미터:

*   `columns`: pivoting된 데이터프레임에서 컬럼이름으로 사용하기 위한 변수(컬럼) 이름(들).
*   `index`: pivoting된 데이터프레임에서 인덱스로 사용하기 위한 변수(컬럼) 이름(들).
*   `values`: pivoting된 데이터프레임에서 각 셀을 채울수 있는 값들을 가지고 있는 변수(컬럼) 이름(들) 연속형변수사용하는듯

In [20]:
df.pivot(columns='day',index='time',values='tip')

day,Fri,Sat,Sun
time,,,
Dinner,4,5,6
Lunch,1,2,3


In [22]:
df.pivot(columns='time',index='day',values='total_bill')

time,Dinner,Lunch
day,,
Fri,40,10
Sat,50,20
Sun,60,30


## melt

`pd.DataFrame.melt()` 메서드 파라미터:

*   `id_vars`: melting 될 때 컬럼으로 유지될 변수 이름(들).
    *   `id_vars`에 설정하지 않은 변수 이름들은 **variable 컬럼**의 값들로 melting됨.
    *   `id_vars`에 설정하지 않은 변수의 값들은 **value 컬럼**의 값들로 melting됨.
*   `var_name`: variable 컬럼의 이름을 대체할 문자열. [옵션.]
*   `value_name`: value 컬럼의 이름을 대체할 문자열.  [옵션.]


In [23]:
df = pd.DataFrame(data={
    'gender':['Female','Male'],
    'lunch':[1,5],
    'dinner':[10,20]

})
df

,gender,lunch,dinner
0,Female,1,10
1,Male,5,20


In [25]:
df.melt(id_vars='gender')

,gender,variable,value
0,Female,lunch,1
1,Male,lunch,5
2,Female,dinner,10
3,Male,dinner,20


In [26]:
df.melt(id_vars='gender',var_name='time',value_name='size')

,gender,time,size
0,Female,lunch,1
1,Male,lunch,5
2,Female,dinner,10
3,Male,dinner,20


# pivot_table

groupby 기능과 통계 함수 적용 결과를 unstack하는 함수.

`pd.DataFrame.pivot_table()` 메서드 파라미터:

*   `values`: 집계(통계) 함수를 적용할 값들을 가지고 있는 변수 이름(들).
*   `index`: 피벗 테이블의 인덱스로 사용할 값들을 가지고 있는 변수 이름(들).
*   `columns`: 피벗 테이블의 컬럼 이름으로 사용할 값들을 가지고 있는 변수 이름(들).
*   `aggfunc`: aggregation function(집계/통계 함수). 기본값은 'mean'.

In [28]:
tips=sns.load_dataset('tips')
tips

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4
...,...,...,...,...,...,...,...
239,29.03,5.92,Male,No,Sat,Dinner,3
240,27.18,2.00,Female,Yes,Sat,Dinner,2
241,22.67,2.00,Male,Yes,Sat,Dinner,2
242,17.82,1.75,Male,No,Sat,Dinner,2


## 성별 팁의 평균

In [35]:
by_sex=tips.groupby(by=['sex'],observed=True).tip.mean()
by_sex

,tip
sex,
Male,3.089618
Female,2.833448


In [37]:
tips.pivot_table(values='tip',index='sex',observed=True) # aggfunc='mean' : 기본값. 생략가능.

,tip
sex,
Male,3.089618
Female,2.833448


## 성별, 흡연여부 별 팁의 평균

In [39]:
by_sex_smoker= tips.groupby(by=['sex','smoker'],observed=True).tip.mean()
by_sex_smoker

sex     smoker
Male    Yes       3.051167
        No        3.113402
Female  Yes       2.931515
        No        2.773519
Name: tip, dtype: float64

In [40]:
by_sex_smoker.unstack()

smoker,Yes,No
sex,,
Male,3.051167,3.113402
Female,2.931515,2.773519


In [44]:
by_sex_smoker.unstack(level=0)

sex,Male,Female
smoker,,
Yes,3.051167,2.931515
No,3.113402,2.773519


In [41]:
tips.pivot_table(values='tip',index=['sex','smoker'],observed=True)

tip
sex    smoker          
Male   Yes     3.051167
       No      3.113402
Female Yes     2.931515
       No      2.773519

In [43]:
tips.pivot_table(values='tip',index='sex',columns='smoker',observed=True)

smoker,Yes,No
sex,,
Male,3.051167,3.113402
Female,2.931515,2.773519


In [45]:
tips.pivot_table(values='tip',index='smoker',columns='sex',observed=True)

sex,Male,Female
smoker,,
Yes,3.051167,2.931515
No,3.113402,2.773519


## 성별 팁과 영수증의 평균

In [47]:
# tips.tip.mean()
tips[['tip','total_bill']].mean()

,0
tip,2.998279
total_bill,19.785943


In [50]:
by_sex=tips.groupby(by=['sex'],observed=True)[['tip','total_bill']].mean() #인덱스 컬럼
by_sex

,tip,total_bill
sex,,
Male,3.089618,20.744076
Female,2.833448,18.056897


In [52]:
tips.pivot_table(values=['tip','total_bill'],index='sex',observed=True)

,tip,total_bill
sex,,
Male,3.089618,20.744076
Female,2.833448,18.056897


## 성별 흡연여부별 팁과 영수증의 평균

In [90]:
by_sex_smoker=tips.groupby(by=['sex','smoker'],observed=True)[['tip','total_bill']].mean()
by_sex_smoker

tip  total_bill
sex    smoker                      
Male   Yes     3.051167   22.284500
       No      3.113402   19.791237
Female Yes     2.931515   17.977879
       No      2.773519   18.105185

In [91]:
by_sex_smoker.unstack()

tip           total_bill           
smoker       Yes        No        Yes         No
sex                                             
Male    3.051167  3.113402  22.284500  19.791237
Female  2.931515  2.773519  17.977879  18.105185

In [92]:
by_sex_smoker.unstack(level=0)

tip           total_bill           
sex         Male    Female       Male     Female
smoker                                          
Yes     3.051167  2.931515  22.284500  17.977879
No      3.113402  2.773519  19.791237  18.105185

In [53]:
tips.pivot_table(values=['tip','total_bill'],index=['sex','smoker'],observed=True) #index(세로) or columns(가로)

tip  total_bill
sex    smoker                      
Male   Yes     3.051167   22.284500
       No      3.113402   19.791237
Female Yes     2.931515   17.977879
       No      2.773519   18.105185

In [93]:
tips.pivot_table(values=['tip','total_bill'],index=['sex'],columns=['smoker'],observed=True)

tip           total_bill           
smoker       Yes        No        Yes         No
sex                                             
Male    3.051167  3.113402  22.284500  19.791237
Female  2.931515  2.773519  17.977879  18.105185

In [94]:
tips.pivot_table(values=['tip','total_bill'],index=['smoker'],columns=['sex'],observed=True)

tip           total_bill           
sex         Male    Female       Male     Female
smoker                                          
Yes     3.051167  2.931515  22.284500  17.977879
No      3.113402  2.773519  19.791237  18.105185

## 성별, 요일별, 시간별 팁의 평균

In [95]:
by_sex_day_time=tips.groupby(by=['sex','day','time'],observed=True)['tip'].mean()
by_sex_day_time

sex     day   time  
Male    Thur  Lunch     2.980333
        Fri   Lunch     1.900000
              Dinner    3.032857
        Sat   Dinner    3.083898
        Sun   Dinner    3.220345
Female  Thur  Lunch     2.561935
              Dinner    3.000000
        Fri   Lunch     2.745000
              Dinner    2.810000
        Sat   Dinner    2.801786
        Sun   Dinner    3.367222
Name: tip, dtype: float64

In [96]:
by_sex_day_time.unstack()

time            Lunch    Dinner
sex    day                     
Male   Thur  2.980333       NaN
       Fri   1.900000  3.032857
       Sat        NaN  3.083898
       Sun        NaN  3.220345
Female Thur  2.561935  3.000000
       Fri   2.745000  2.810000
       Sat        NaN  2.801786
       Sun        NaN  3.367222

In [97]:
by_sex_day_time.unstack(level=1)

day                Thur       Fri       Sat       Sun
sex    time                                          
Male   Lunch   2.980333  1.900000       NaN       NaN
       Dinner       NaN  3.032857  3.083898  3.220345
Female Lunch   2.561935  2.745000       NaN       NaN
       Dinner  3.000000  2.810000  2.801786  3.367222

In [98]:
by_sex_day_time.unstack(level=0)

sex              Male    Female
day  time                      
Thur Lunch   2.980333  2.561935
     Dinner       NaN  3.000000
Fri  Lunch   1.900000  2.745000
     Dinner  3.032857  2.810000
Sat  Dinner  3.083898  2.801786
Sun  Dinner  3.220345  3.367222

In [100]:
# by_sex_day_time.unstack(level=[1,2])
by_sex_day_time.unstack(level=[2,1])

time       Lunch           Dinner                         
day         Thur    Fri       Fri       Sat       Sun Thur
sex                                                       
Male    2.980333  1.900  3.032857  3.083898  3.220345  NaN
Female  2.561935  2.745  2.810000  2.801786  3.367222  3.0

In [55]:
tips.pivot_table(values=['tip'],index=['sex','day','time'],observed=True)

tip
sex    day  time            
Male   Thur Lunch   2.980333
       Fri  Lunch   1.900000
            Dinner  3.032857
       Sat  Dinner  3.083898
       Sun  Dinner  3.220345
Female Thur Lunch   2.561935
            Dinner  3.000000
       Fri  Lunch   2.745000
            Dinner  2.810000
       Sat  Dinner  2.801786
       Sun  Dinner  3.367222

In [101]:
tips.pivot_table(values=['tip'],columns=['sex','day','time'],observed=True)

sex       Male                                        Female                \
day       Thur   Fri                 Sat       Sun      Thur           Fri   
time     Lunch Lunch    Dinner    Dinner    Dinner     Lunch Dinner  Lunch   
tip   2.980333   1.9  3.032857  3.083898  3.220345  2.561935    3.0  2.745   

sex                              
day               Sat       Sun  
time Dinner    Dinner    Dinner  
tip    2.81  2.801786  3.367222

In [102]:
tips.pivot_table(values=['tip'],index=['day','time'],columns=['sex'],observed=True)

tip          
sex              Male    Female
day  time                      
Thur Lunch   2.980333  2.561935
     Dinner       NaN  3.000000
Fri  Lunch   1.900000  2.745000
     Dinner  3.032857  2.810000
Sat  Dinner  3.083898  2.801786
Sun  Dinner  3.220345  3.367222

In [103]:
tips.pivot_table(values=['tip'],columns=['day','time'],index=['sex'],observed=True)

tip                                            
day         Thur           Fri                 Sat       Sun
time       Lunch Dinner  Lunch    Dinner    Dinner    Dinner
sex                                                         
Male    2.980333    NaN  1.900  3.032857  3.083898  3.220345
Female  2.561935    3.0  2.745  2.810000  2.801786  3.367222

## 성별, 팁의 최솟값, 중앙값, 최댓값

In [104]:
result=tips.groupby(by=['sex'],observed=True).tip.agg(['min','median','max'])
result                                            #aggregate 메서드 사용.

,min,median,max
sex,,,
Male,1.0,3.00,10.0
Female,1.0,2.75,6.5


In [59]:
tips.pivot_table(values=['tip'],index=['sex'],observed=True,aggfunc={'tip':['min','median','max']})
                                                          # aggfunc=['min','median','max'] 이것도 가능

tip            
         max median  min
sex                     
Male    10.0   3.00  1.0
Female   6.5   2.75  1.0

## 성별, 요일별 영수증 최솟값, 중앙값, 최댓값

In [106]:
result=tips.groupby(by=['sex','day'],observed=True).total_bill.agg(['min','median','max'])
result

min  median    max
sex    day                      
Male   Thur  7.51  16.975  41.19
       Fri   8.58  17.215  40.17
       Sat   7.74  18.240  50.81
       Sun   7.25  20.725  48.17
Female Thur  8.35  13.785  43.11
       Fri   5.75  15.380  22.75
       Sat   3.07  18.360  44.30
       Sun   9.60  17.410  35.26

In [107]:
result.unstack()

min                    median                           max         \
day     Thur   Fri   Sat   Sun    Thur     Fri    Sat     Sun   Thur    Fri   
sex                                                                           
Male    7.51  8.58  7.74  7.25  16.975  17.215  18.24  20.725  41.19  40.17   
Female  8.35  5.75  3.07  9.60  13.785  15.380  18.36  17.410  43.11  22.75   

                      
day       Sat    Sun  
sex                   
Male    50.81  48.17  
Female  44.30  35.26

In [60]:
tips.pivot_table(values=['total_bill'],index=['sex','day'],observed=True,aggfunc={'total_bill':['min','median','max']})

total_bill              
                   max  median   min
sex    day                          
Male   Thur      41.19  16.975  7.51
       Fri       40.17  17.215  8.58
       Sat       50.81  18.240  7.74
       Sun       48.17  20.725  7.25
Female Thur      43.11  13.785  8.35
       Fri       22.75  15.380  5.75
       Sat       44.30  18.360  3.07
       Sun       35.26  17.410  9.60

In [108]:
tips.pivot_table(values=['total_bill'],index=['sex'],columns=['day'],observed=True,aggfunc={'total_bill':['min','median','max']})

total_bill                                                            \
              max                       median                          min   
day          Thur    Fri    Sat    Sun    Thur     Fri    Sat     Sun  Thur   
sex                                                                           
Male        41.19  40.17  50.81  48.17  16.975  17.215  18.24  20.725  7.51   
Female      43.11  22.75  44.30  35.26  13.785  15.380  18.36  17.410  8.35   

                          
                          
day      Fri   Sat   Sun  
sex                       
Male    8.58  7.74  7.25  
Female  5.75  3.07  9.60

In [109]:
tips.pivot_table(values=['total_bill'],index=['day'],columns=['sex'],observed=True,aggfunc={'total_bill':['min','median','max']})

total_bill                                    
            max         median           min       
sex        Male Female    Male  Female  Male Female
day                                                
Thur      41.19  43.11  16.975  13.785  7.51   8.35
Fri       40.17  22.75  17.215  15.380  8.58   5.75
Sat       50.81  44.30  18.240  18.360  7.74   3.07
Sun       48.17  35.26  20.725  17.410  7.25   9.60

## 성별, 흡연여부별, 요일별 팁의 중앙값

In [110]:
result=tips.groupby(by=['sex','smoker','day'],observed=True).tip.agg(['median'])
result

median
sex    smoker day         
Male   Yes    Thur   2.780
              Fri    2.600
              Sat    3.000
              Sun    3.500
       No     Thur   2.405
              Fri    2.500
              Sat    2.860
              Sun    3.000
Female Yes    Thur   2.500
              Fri    2.500
              Sat    2.500
              Sun    3.500
       No     Thur   2.000
              Fri    3.125
              Sat    2.750
              Sun    3.500

In [115]:
result.unstack()

median
sex    smoker day         
Male   Yes    Thur   2.780
              Fri    2.600
              Sat    3.000
              Sun    3.500
       No     Thur   2.405
              Fri    2.500
              Sat    2.860
              Sun    3.000
Female Yes    Thur   2.500
              Fri    2.500
              Sat    2.500
              Sun    3.500
       No     Thur   2.000
              Fri    3.125
              Sat    2.750
              Sun    3.500

In [116]:
result.unstack(level=[0,1])

median                     
sex      Male        Female       
smoker    Yes     No    Yes     No
day                               
Thur     2.78  2.405    2.5  2.000
Fri      2.60  2.500    2.5  3.125
Sat      3.00  2.860    2.5  2.750
Sun      3.50  3.000    3.5  3.500

In [61]:
tips.pivot_table(values=['tip'],index=['sex','smoker','day'],observed=True,aggfunc={'tip':'median'})

tip
sex    smoker day        
Male   Yes    Thur  2.780
              Fri   2.600
              Sat   3.000
              Sun   3.500
       No     Thur  2.405
              Fri   2.500
              Sat   2.860
              Sun   3.000
Female Yes    Thur  2.500
              Fri   2.500
              Sat   2.500
              Sun   3.500
       No     Thur  2.000
              Fri   3.125
              Sat   2.750
              Sun   3.500

In [117]:
tips.pivot_table(values=['tip'],index=['sex','smoker'],columns=['day'],observed=True,aggfunc={'tip':'median'})

tip                  
day             Thur    Fri   Sat  Sun
sex    smoker                         
Male   Yes     2.780  2.600  3.00  3.5
       No      2.405  2.500  2.86  3.0
Female Yes     2.500  2.500  2.50  3.5
       No      2.000  3.125  2.75  3.5

In [118]:
tips.pivot_table(values=['tip'],columns=['sex','smoker'],index=['day'],observed=True,aggfunc={'tip':'median'})

tip                     
sex     Male        Female       
smoker   Yes     No    Yes     No
day                              
Thur    2.78  2.405    2.5  2.000
Fri     2.60  2.500    2.5  3.125
Sat     3.00  2.860    2.5  2.750
Sun     3.50  3.000    3.5  3.500

## 성별, 흡연여부별, 요일별, 시간별 팁의 중앙값

In [120]:
result=tips.groupby(by=['sex','smoker','day','time'],observed=True).tip.agg(['median'])#.unstack(level=[-4])
result

median
sex    smoker day  time          
Male   Yes    Thur Lunch    2.780
              Fri  Lunch    1.920
                   Dinner   3.000
              Sat  Dinner   3.000
              Sun  Dinner   3.500
       No     Thur Lunch    2.405
              Fri  Dinner   2.500
              Sat  Dinner   2.860
              Sun  Dinner   3.000
Female Yes    Thur Lunch    2.500
              Fri  Lunch    2.500
                   Dinner   2.750
              Sat  Dinner   2.500
              Sun  Dinner   3.500
       No     Thur Lunch    2.000
                   Dinner   3.000
              Fri  Lunch    3.000
                   Dinner   3.250
              Sat  Dinner   2.750
              Sun  Dinner   3.500

In [121]:
result.unstack(level=[2,3])

median                                  
day             Thur   Fri           Sat    Sun   Thur
time           Lunch Lunch Dinner Dinner Dinner Dinner
sex    smoker                                         
Male   Yes     2.780  1.92   3.00   3.00    3.5    NaN
       No      2.405   NaN   2.50   2.86    3.0    NaN
Female Yes     2.500  2.50   2.75   2.50    3.5    NaN
       No      2.000  3.00   3.25   2.75    3.5    3.0

In [122]:
result.unstack(level=[0,1])

median                    
sex           Male        Female      
smoker         Yes     No    Yes    No
day  time                             
Thur Lunch    2.78  2.405   2.50  2.00
     Dinner    NaN    NaN    NaN  3.00
Fri  Lunch    1.92    NaN   2.50  3.00
     Dinner   3.00  2.500   2.75  3.25
Sat  Dinner   3.00  2.860   2.50  2.75
Sun  Dinner   3.50  3.000   3.50  3.50

In [111]:
tips.pivot_table(values=['tip'],index=['sex','smoker','day','time'],observed=True,aggfunc={'tip':'median'})#.unstack(level=[-4])

tip
sex    smoker day  time         
Male   Yes    Thur Lunch   2.780
              Fri  Lunch   1.920
                   Dinner  3.000
              Sat  Dinner  3.000
              Sun  Dinner  3.500
       No     Thur Lunch   2.405
              Fri  Dinner  2.500
              Sat  Dinner  2.860
              Sun  Dinner  3.000
Female Yes    Thur Lunch   2.500
              Fri  Lunch   2.500
                   Dinner  2.750
              Sat  Dinner  2.500
              Sun  Dinner  3.500
       No     Thur Lunch   2.000
                   Dinner  3.000
              Fri  Lunch   3.000
                   Dinner  3.250
              Sat  Dinner  2.750
              Sun  Dinner  3.500

In [123]:
tips.pivot_table(values=['tip'],index=['sex','smoker'],columns=['day','time'],observed=True,aggfunc={'tip':'median'})#.unstack(level=[-4])

tip                                  
day             Thur          Fri           Sat    Sun
time           Lunch Dinner Lunch Dinner Dinner Dinner
sex    smoker                                         
Male   Yes     2.780    NaN  1.92   3.00   3.00    3.5
       No      2.405    NaN   NaN   2.50   2.86    3.0
Female Yes     2.500    NaN  2.50   2.75   2.50    3.5
       No      2.000    3.0  3.00   3.25   2.75    3.5

In [124]:
tips.pivot_table(values=['tip'],columns=['sex','smoker'],index=['day','time'],observed=True,aggfunc={'tip':'median'})#.unstack(level=[-4])

tip                    
sex          Male        Female      
smoker        Yes     No    Yes    No
day  time                            
Thur Lunch   2.78  2.405   2.50  2.00
     Dinner   NaN    NaN    NaN  3.00
Fri  Lunch   1.92    NaN   2.50  3.00
     Dinner  3.00  2.500   2.75  3.25
Sat  Dinner  3.00  2.860   2.50  2.75
Sun  Dinner  3.50  3.000   3.50  3.50